## Estructura del AST

El Abstract Syntax Tree es una representacion jerarquica del codigo pseudocodigo.
Cada nodo del arbol representa una construccion sintactica del lenguaje.

### Nodo Raiz: ProgramNode

Todo AST comienza con un `ProgramNode` que contiene:

```
ProgramNode
|
+-- classes: List[ClassDefinitionNode]  # Definiciones de clases (opcional)
|
+-- algorithm: AlgorithmNode            # Algoritmo principal
```

### Nodo de Algoritmo: AlgorithmNode

Representa la definicion del algoritmo:

```
AlgorithmNode
|
+-- name: str                           # Nombre del algoritmo
|
+-- parameters: List[ParameterNode]     # Lista de parametros
|
+-- body: BlockNode                     # Cuerpo del algoritmo
```

## Jerarquia de Nodos

### Diagrama de Herencia

Todos los nodos heredan de la clase base abstracta `ASTNode`:

```
ASTNode (abstracta)
|
+-- ProgramNode
|
+-- ClassDefinitionNode
|
+-- AlgorithmNode
|
+-- ParameterNode
|
+-- BlockNode
|
+-- Statements
|   +-- AssignmentNode
|   +-- ForLoopNode
|   +-- WhileLoopNode
|   +-- RepeatLoopNode
|   +-- IfStatementNode
|   +-- CallStatementNode
|   +-- ReturnStatementNode
|
+-- Expressions
    +-- LiteralNode
    +-- VariableNode
    +-- BinaryOpNode
    +-- UnaryOpNode
    +-- ArrayAccessNode
    +-- ObjectAccessNode
    +-- FunctionCallNode
```

### Atributos Comunes (ASTNode)

Todos los nodos tienen estos atributos:

| Atributo | Tipo | Descripcion |
|----------|------|-------------|
| `node_type` | `ASTNodeType` | Enumeracion del tipo de nodo |
| `line` | `Optional[int]` | Linea en el codigo fuente |
| `column` | `Optional[int]` | Columna en el codigo fuente |

## Navegacion del AST

### Estructura de un Ciclo For

El nodo `ForLoopNode` tiene la siguiente estructura:

```
ForLoopNode
|
+-- variable: str        # Variable de iteracion (ej: "i")
|
+-- start: ExpressionNode # Valor inicial
|
+-- end: ExpressionNode   # Valor final
|
+-- body: BlockNode       # Cuerpo del ciclo
```

### Estructura de un Condicional If

```
IfStatementNode
|
+-- condition: ExpressionNode  # Condicion booleana
|
+-- then_block: BlockNode      # Bloque si verdadero
|
+-- else_block: BlockNode      # Bloque si falso (opcional)
```

### Estructura de una Asignacion

```
AssignmentNode
|
+-- target: LValueNode         # Lado izquierdo (variable)
|
+-- value: ExpressionNode      # Lado derecho (expresion)
```

## Representacion en Diccionario

Cada nodo del AST puede convertirse a diccionario usando el metodo `to_dict()`.
Esto es util para:

- Serializacion JSON
- Depuracion
- Exportacion

### Ejemplo de Estructura JSON

Para un algoritmo simple:

```
algorithm suma(n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + i
    end
    return total
end
```

El diccionario resultante seria:

```json
{
  "type": "program",
  "classes": [],
  "algorithm": {
    "type": "algorithm",
    "name": "suma",
    "parameters": [
      {"type": "parameter", "name": "n", "param_type": "simple"}
    ],
    "body": {
      "type": "block",
      "statements": [
        {
          "type": "assignment",
          "target": {"name": "total"},
          "value": {"type": "literal", "value": 0}
        },
        {
          "type": "for_loop",
          "variable": "i",
          "start": {"type": "literal", "value": 1},
          "end": {"type": "variable", "name": "n"},
          "body": {...}
        },
        {
          "type": "return",
          "value": {"type": "variable", "name": "total"}
        }
      ]
    }
  }
}
```

## Ejemplos de Visualizacion

### Ejemplo 1: Algoritmo de Busqueda Binaria

Codigo:

```
algorithm binarySearch(A[], n, key)
begin
    low <- 1
    high <- n
    while (low <= high) do
        mid <- floor((low + high) / 2)
        if (A[mid] = key) then
            return mid
        else
            if (A[mid] < key) then
                low <- mid + 1
            else
                high <- mid - 1
            end
        end
    end
    return -1
end
```

Visualizacion del AST:

```
ProgramNode
|
+-- AlgorithmNode("binarySearch")
    |
    +-- Parameters: [A[], n, key]
    |
    +-- BlockNode
        |
        +-- AssignmentNode(low <- 1)
        |
        +-- AssignmentNode(high <- n)
        |
        +-- WhileLoopNode(low <= high)
        |   |
        |   +-- BlockNode
        |       |
        |       +-- AssignmentNode(mid <- floor(...))
        |       |
        |       +-- IfStatementNode(A[mid] = key)
        |           |
        |           +-- then: ReturnStatementNode(mid)
        |           |
        |           +-- else: IfStatementNode(A[mid] < key)
        |                     +-- then: AssignmentNode(low <- mid + 1)
        |                     +-- else: AssignmentNode(high <- mid - 1)
        |
        +-- ReturnStatementNode(-1)
```

### Ejemplo 2: Algoritmo Recursivo (Fibonacci)

Codigo:

```
algorithm fibonacci(n)
begin
    if (n <= 1) then
        return n
    else
        return call fibonacci(n - 1) + call fibonacci(n - 2)
    end
end
```

Visualizacion del AST:

```
ProgramNode
|
+-- AlgorithmNode("fibonacci")
    |
    +-- Parameters: [n]
    |
    +-- BlockNode
        |
        +-- IfStatementNode(n <= 1)
            |
            +-- then: BlockNode
            |         +-- ReturnStatementNode(n)
            |
            +-- else: BlockNode
                      +-- ReturnStatementNode
                          |
                          +-- BinaryOpNode(+)
                              |
                              +-- CallStatementNode(fibonacci, n-1)
                              |
                              +-- CallStatementNode(fibonacci, n-2)
```

### Ejemplo 3: Ciclos Anidados (Bubble Sort)

Codigo:

```
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
```

Visualizacion del AST:

```
ProgramNode
|
+-- AlgorithmNode("bubbleSort")
    |
    +-- Parameters: [A[], n]
    |
    +-- BlockNode
        |
        +-- ForLoopNode(i: 1 to n-1)
            |
            +-- BlockNode
                |
                +-- ForLoopNode(j: 1 to n-i)
                    |
                    +-- BlockNode
                        |
                        +-- IfStatementNode(A[j] > A[j+1])
                            |
                            +-- then: BlockNode
                                      +-- AssignmentNode(temp <- A[j])
                                      +-- AssignmentNode(A[j] <- A[j+1])
                                      +-- AssignmentNode(A[j+1] <- temp)
```

Esta estructura anidada es importante para el analisis de complejidad:
- El ciclo externo itera n-1 veces
- El ciclo interno itera n-i veces
- Resultado: O(n^2)

## Demostracion: Visualizacion Programatica del AST

In [ ]:
import sys
import json
sys.path.insert(0, '../..')

from app.core.parser.pseudocode_parser import PseudocodeParser

def visualize_ast(node, indent=0):
    """Funcion recursiva para visualizar el AST"""
    prefix = "  " * indent
    print(f"{prefix}{node.__class__.__name__}")
    
    if hasattr(node, 'name'):
        print(f"{prefix}  name: {node.name}")
    
    if hasattr(node, 'statements'):
        for stmt in node.statements:
            visualize_ast(stmt, indent + 1)
    
    if hasattr(node, 'body') and node.body:
        visualize_ast(node.body, indent + 1)

# Parsear codigo de ejemplo
parser = PseudocodeParser()
codigo = '''
algorithm ejemplo(n)
begin
    for i <- 1 to n do
        x <- x + 1
    end
end
'''

ast = parser.parse(codigo)

# Visualizar estructura
print("=== Estructura del AST ===")
visualize_ast(ast.algorithm)

# Mostrar como diccionario
print("\n=== Representacion JSON ===")
print(json.dumps(ast.to_dict(), indent=2))

---

## Conclusiones

El AST proporciona una representacion estructurada del codigo que permite:

1. **Analisis de complejidad**: Identificar ciclos anidados, recursion y estructuras de control
2. **Deteccion de patrones**: Reconocer estructuras como divide y venceras, backtracking, etc.
3. **Transformaciones**: Modificar o generar codigo basado en la estructura
4. **Visualizacion**: Crear diagramas de flujo y arboles de recursion

---

**Siguiente notebook recomendado**: `grammar_experiments.ipynb` para explorar la gramatica Lark.